In [5]:
import pandas as pd
import os
import numpy as np

group_cols = ["User", "Model", "Device"]

unmerged_acc = pd.read_csv('unmerged_acc_onarrivaltime.csv')
unmerged_gyro = pd.read_csv('unmerged_gyro_onarrivaltime.csv')
merged_df = pd.read_csv('merged_data_onarrivaltime.csv')

merged_df.head()

/var/folders/s2/_qxs1d_57r18ktd8jtkpm0sw0000gn/T/ipykernel_80508/433928640.py:9: DtypeWarning: Columns (8,14) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_df = pd.read_csv('merged_data_onarrivaltime.csv')


,Arrival_Time,Creation_Time_x,x_x,y_x,z_x,User,Model,Device,gt_x,acc_index,Creation_Time_y,x_y,y_y,z_y,gt_y,gyro_index
0,1424696638743,27920853362000,-9.273342,-3.729568,0.240617,a,gear,gear_1,stand,13,27920678496000,-0.162187,-0.022104,0.059655,stand,0
1,1424696638743,27920853362000,-9.273342,-3.729568,0.240617,a,gear,gear_1,stand,13,27920681926000,-0.183225,-0.061785,0.012517,stand,1
2,1424696638743,27920853362000,-9.273342,-3.729568,0.240617,a,gear,gear_1,stand,13,27920692031000,-0.180829,-0.108657,-0.036485,stand,2
3,1424696638743,27920853362000,-9.273342,-3.729568,0.240617,a,gear,gear_1,stand,13,27920701997000,-0.147805,-0.157925,-0.098537,stand,3
4,1424696639000,27921599563000,-9.153033,-3.605668,-0.732626,a,gear,gear_1,stand,15,27921788680000,-0.023968,-0.067911,-0.091613,stand,29


In [ ]:

print("Unmerged_acc count:", len(unmerged_acc))
print("Unmerged_gyro count:", len(unmerged_gyro))
print("Merged_df count:", len(merged_df))

# 2. Extract unique combinations from merged_df
unique_groups = merged_df[group_cols].drop_duplicates().reset_index(drop=True)

# 3. Calculate row counts per group across datasets
merged_counts = merged_df.groupby(group_cols).size().rename("merged_count")
acc_counts = unmerged_acc.groupby(group_cols).size().rename("unmerged_acc_count")
gyro_counts = unmerged_gyro.groupby(group_cols).size().rename("unmerged_gyro_count")

# 4. Join counts into a single inspection table
group_breakdown = (
    unique_groups
    .set_index(group_cols)
    .join(merged_counts, how='left')
    .join(acc_counts, how='left')
    .join(gyro_counts, how='left')
    .fillna(0)
    .astype(int)
    .reset_index()
)

# 5. Identify incomplete groups (unmerged count > 0 in either dataset)
group_breakdown['is_incomplete'] = (group_breakdown['unmerged_acc_count'] > 0) | (group_breakdown['unmerged_gyro_count'] > 0)

incomplete_groups = group_breakdown[group_breakdown['is_incomplete']]

# 6. Summary checks
print(f"Total groups in merged_df: {len(group_breakdown)}")
print(f"Incomplete groups in merged_df: {len(incomplete_groups)}")

print("\n--- Incomplete Groups Breakdown ---")
print(incomplete_groups)

In [4]:
df_acc = pd.read_csv('../ignore_files/data/full/Phones_accelerometer.csv')
df_gyro = pd.read_csv('../ignore_files/data/full/Phones_gyroscope.csv')

# 1. Extract unique groups from each DataFrame
acc_groups = df_acc[group_cols].drop_duplicates()
gyro_groups = df_gyro[group_cols].drop_duplicates()

# 2. Combine and drop duplicates to get all distinct groups
all_groups = pd.concat([acc_groups, gyro_groups]).drop_duplicates().reset_index(drop=True)

print(f"Total unique groups across both datasets: {len(all_groups)}")
print(all_groups)

Total unique groups across both datasets: 72
   User       Model        Device
0     a      nexus4      nexus4_1
1     a      nexus4      nexus4_2
2     a          s3          s3_1
3     a          s3          s3_2
4     a      s3mini      s3mini_1
..  ...         ...           ...
67    i      s3mini      s3mini_1
68    i      s3mini      s3mini_2
69    i  samsungold  samsungold_1
70    i  samsungold  samsungold_2
71    f          s3          s3_1

[72 rows x 3 columns]


In [ ]:
def find_missing_ranges_numpy(arr):
    """
    Returns missing integer ranges between 0 and max(arr) inclusive.
    Formats consecutive missing numbers as 'start-end' (e.g., '1-190').
    """
    if len(arr) == 0:
        return []
    
    max_val = int(np.max(arr))
    full_range = np.arange(0, max_val + 1)
    missing = np.setdiff1d(full_range, arr)
    
    if len(missing) == 0:
        return []
    
    # 1. Identify indices where the difference is greater than 1 (gaps in sequence)
    splits = np.where(np.diff(missing) != 1)[0] + 1
    
    # 2. Split into contiguous blocks of missing numbers
    sub_ranges = np.split(missing, splits)
    
    # 3. Format as "start-end" or single number strings
    result = [
        f"{r[0]}-{r[-1]}" if len(r) > 1 else str(r[0])
        for r in sub_ranges
    ]
    
    return result


